In [ ]:
! pip install benepar
! pip install nltk zss

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [5]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
import benepar
benepar.download('benepar_en3')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\skpaul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\skpaul\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package benepar_en3 to
[nltk_data]     C:\Users\skpaul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping models\benepar_en3.zip.


True

In [8]:
import nltk
from nltk.tree import Tree
import benepar
from zss import simple_distance, Node

# Load the benepar parser
parser = benepar.Parser("benepar_en3")

def sentence_to_tree(sentence):
    """Parse a sentence into an NLTK Tree using benepar."""
    sent = nltk.word_tokenize(sentence)
    tree = parser.parse(sent)
    return tree

def nltk_tree_to_zss(tree):
    """Convert NLTK Tree to zss.Node for tree edit distance computation."""
    node = Node(str(tree.label()))
    for child in tree:
        if isinstance(child, Tree):
            node.addkid(nltk_tree_to_zss(child))
        else:
            # For leaves, add as a child node
            node.addkid(Node(str(child)))
    return node

def compute_tree_edit_distance(tree1, tree2):
    """Compute tree edit distance between two NLTK Trees."""
    node1 = nltk_tree_to_zss(tree1)
    node2 = nltk_tree_to_zss(tree2)
    return simple_distance(node1, node2)

def draw_tree(tree, title="Parse Tree"):
    """Draw the NLTK parse tree."""
    print(title)
    tree.pretty_print()

def process_sentence_pairs(pairs):
    """
    pairs: list of (original, back_translated) tuples
    """
    results = []
    for idx, (orig, aug) in enumerate(pairs):
        print(f"\nPair {idx+1}:")
        print("Original:      ", orig)
        print("Back-translated:", aug)
        try:
            tree_orig = sentence_to_tree(orig)
            tree_aug = sentence_to_tree(aug)
            dist = compute_tree_edit_distance(tree_orig, tree_aug)
            print("Tree Edit Distance:", dist)
            print("Original Parse Tree:")
            draw_tree(tree_orig)
            print("Back-Translated Parse Tree:")
            draw_tree(tree_aug)
            results.append({'pair': (orig, aug), 'distance': dist})
        except Exception as e:
            print("Error parsing or computing distance:", e)
            results.append({'pair': (orig, aug), 'distance': None})
    return results

# Example input
sentence_pairs = [
    ("The cat sat on the mat.", "The mat was sat on by the cat."),
    ("She enjoys reading books.", "She likes to read books."),
    # Add more pairs as needed
]

# Run the process
results = process_sentence_pairs(sentence_pairs)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Pair 1:
Original:       The cat sat on the mat.
Back-translated: The mat was sat on by the cat.
Tree Edit Distance: 10.0
Original Parse Tree:
Parse Tree
                TOP                    
                 |                      
                 S                     
      ___________|___________________   
     |               VP              | 
     |        _______|___            |  
     |       |           PP          | 
     |       |    _______|___        |  
     NP      |   |           NP      | 
  ___|___    |   |        ___|___    |  
 DT      NN VBD  IN      DT      NN  . 
 |       |   |   |       |       |   |  
The     cat sat  on     the     mat  . 

Back-Translated Parse Tree:
Parse Tree
                    TOP                    
                     |                      
                     S                     
      _______________|___________________   
     |               VP                  | 
     |        _______|___                |  
     |       

In [ ]:
import pandas as pd
import nltk
from nltk.tree import Tree
from zss import simple_distance, Node

# Ensure all required NLTK resources are downloaded
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Import and download benepar model
import benepar
benepar.download('benepar_en3')
parser = benepar.Parser("benepar_en3")

# Utility functions
def sentence_to_tree(sentence):
    """Parse a sentence into an NLTK Tree using benepar."""
    tokens = nltk.word_tokenize(str(sentence))
    return parser.parse(tokens)

def nltk_tree_to_zss(tree):
    """Convert NLTK Tree to zss.Node for tree edit distance computation."""
    node = Node(str(tree.label()))
    for child in tree:
        if isinstance(child, Tree):
            node.addkid(nltk_tree_to_zss(child))
        else:
            node.addkid(Node(str(child)))
    return node

def compute_tree_edit_distance(tree1, tree2):
    node1 = nltk_tree_to_zss(tree1)
    node2 = nltk_tree_to_zss(tree2)
    return simple_distance(node1, node2)

# Load your CSV
df = pd.read_csv('data\\dataset_aug_train_all_new.csv')  # Change to your actual file name

# Identify augment columns
augment_columns = [col for col in df.columns if col.startswith('augment_')]

results = []
progress_interval = 10  # Print progress every 10 rows

for idx, row in df.iterrows():
    orig_sentence = row['text']
    try:
        tree_orig = sentence_to_tree(orig_sentence)
    except Exception as e:
        tree_orig = None
    row_result = {'row': idx, 'text': orig_sentence}
    for col in augment_columns:
        aug_sentence = row[col]
        try:
            tree_aug = sentence_to_tree(aug_sentence)
            if tree_orig is not None:
                dist = compute_tree_edit_distance(tree_orig, tree_aug)
            else:
                dist = None
        except Exception as e:
            dist = None
        row_result[col + '_parse_tree_distance'] = dist
    results.append(row_result)
    # Print progress
    if (idx + 1) % progress_interval == 0:
        print(f"Processed {idx + 1} rows out of {len(df)}")

print("Processing complete.")

results_df = pd.DataFrame(results)
results_df.to_csv('results\\parse_tree_distances.csv', index=False)
print(results_df.head())

